In [7]:
from pathlib import Path
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive/NUIN/motion_CNN
    DATA_PATH = Path("/content/drive/MyDrive/NUIN/van_hateren/") 
    !git pull
else: 
    DATA_PATH = Path("/home/guardomayas/NUIN/van_hateren/vanhateren_iml")


In [8]:
import numpy as np
import matplotlib.pyplot as plt
from motionnet.dataset import GetNaturalMovies, animate_sample, plot_examples
from torch.utils.data import DataLoader
from tqdm import tqdm
from IPython.display import HTML, display
from motionnet.dataset.van_hateren_utils import log_image
from pathlib import Path

In [9]:
## GLOBAL VARS
CACHE_DIR = Path("/home/guardomayas/NUIN/motionnet/data/coeffs")
N_IMAGES = 500
RNG      = np.random.default_rng(0)
# DPP      = 1/60 # Van Hateren images are 1/60 deg per pixel 
plot_vid = True


In [16]:
from motionnet.dataset import build_coeff_cache, split_images

CFG = dict(eye_size=28, blur_px=13, start_jitter=100,
           frames_per_segment=75*1.5, fps=75,
           vel_std_deg_s=(120, 70), samples_per_image=20)

RHO_PHI = 1.08                                   # matches the class default
SIGMA_PX = CFG["blur_px"] * RHO_PHI / 2.3548

train_files, val_files = split_images(DATA_PATH, n_images=N_IMAGES,
                                      val_frac=0.2, stride=5)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
train_cache = CACHE_DIR / f"train_n{len(train_files)}_blur{CFG['blur_px']}.npy"
val_cache   = CACHE_DIR / f"val_n{len(val_files)}_blur{CFG['blur_px']}.npy"

if not train_cache.exists():
    build_coeff_cache(train_files, train_cache, SIGMA_PX)
if not val_cache.exists():
    build_coeff_cache(val_files, val_cache, SIGMA_PX)

train_ds = GetNaturalMovies(files=train_files, coeff_cache=train_cache,
                            seed=1, **CFG)
val_ds   = GetNaturalMovies(files=val_files, coeff_cache=val_cache,
                            seed=99, **CFG)

/tmp/ipykernel_272368/3929230254.py:22: RuntimeWarning: excursion budget is tight on x: limit 475 px is 1.9 sigma of the position spread (246 px). In 'reject' mode this means many redraws. Reduce vel_std_deg_s (now (120.0, 70.0)), frames_per_segment (now 112), or start_jitter (now 100).
  train_ds = GetNaturalMovies(files=train_files, coeff_cache=train_cache,
/tmp/ipykernel_272368/3929230254.py:22: RuntimeWarning: excursion budget is tight on y: limit 219 px is 1.5 sigma of the position spread (143 px). In 'reject' mode this means many redraws. Reduce vel_std_deg_s (now (120.0, 70.0)), frames_per_segment (now 112), or start_jitter (now 100).
  train_ds = GetNaturalMovies(files=train_files, coeff_cache=train_cache,
/tmp/ipykernel_272368/3929230254.py:24: RuntimeWarning: excursion budget is tight on x: limit 475 px is 1.9 sigma of the position spread (246 px). In 'reject' mode this means many redraws. Reduce vel_std_deg_s (now (120.0, 70.0)), frames_per_segment (now 112), or start_jitter

In [17]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4,
                      pin_memory=True, drop_last=True, persistent_workers=True)
val_dl   = DataLoader(val_ds, batch_size=16, num_workers=2,
                      persistent_workers=True)

In [18]:
b = next(iter(train_dl))
print(b["movie"].shape)                       # (16, 90, 1, 28, 28)
print(b["vel_tap_frame"].std((0, 1)))         # ~(0.30, 0.18)
for i in range(100):
    train_ds[i]
print(f"{100 * train_ds.acceptance_rate:.0f}%")

torch.Size([16, 127, 1, 28, 28])
tensor([0.2779, 0.1450])
84%
